In [1]:
import pandas as pd
from nltk.corpus import stopwords
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import numpy as np

In [2]:
stop_words = set(stopwords.words('english'))

In [3]:
#Nettoyage des données
def text_process(mess):
    lower_mess = mess.lower()
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

In [4]:
train_data = pd.read_csv('../../data/train_unique_e2p2.csv')
test_data = pd.read_csv('../../data/test.csv')

In [5]:
train_data["Context_clean"] = train_data["Context"].apply(text_process)
train_data["Response_clean"] = train_data["Response"].apply(text_process)
# Structure du dictionnaire : mot -> liste des discussions contenant ce type de mot
index_inverse = {}
# Nombre minimum d'occurrences pour garder une discussion
SEUIL = 1
for idx, row in train_data.iterrows():
    question = row["Context"]
    response = row["Response"]
    #combine les mots de la question et de la réponse
    mots = row["Context_clean"] + row["Response_clean"]
    # Compter le nombre d'apparitions de chaque mot
    from collections import Counter
    compteur = Counter(mots)
    for mot,count in compteur.items():
        if count >= SEUIL:
        #Si le mot n'existe pas encore dans le dictionnaire, on l'initialise
            if mot not in index_inverse:
                index_inverse[mot] = []
            #On ajoute la discussion associée à ce mot dans la liste
            index_inverse[mot].append({
                "id": idx,
                "question": question,
                "response": response
            })

In [6]:
anxiete = index_inverse.get("anxiety", [])
print("Nombre de discussion contenant le mot anxiety plus de",SEUIL,"fois :", len(anxiete))
print("\nDiscussion pour le mot 'anxiety' :\n")
#Pour un affichage plus propre
for i, item in enumerate(anxiete, 1):
    print(f"--- Exemple {i} ---")
    print(f"ID       : {item['id']}")
    print(f"Question : {item['question']}")
    print(f"Réponse  : {item['response']}\n")

Nombre de discussion contenant le mot anxiety plus de 1 fois : 114

Discussion pour le mot 'anxiety' :

--- Exemple 1 ---
ID       : 1
Question : I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?
Réponse  : Let me start by saying there are never too many concerns that you can bring into counselling. In fact, most people who come to see me for counselling have more than one issue they would like to work on in psychotherapy and most times these are all interconnected. In counselling, we work together, collaboratively, to figure out which issues you would like to address first and then together we develop an individualized plan of care. Basically, it’s like a road map 

In [7]:
# Méthode 1 : TF-IDF
vectorizer = TfidfVectorizer(analyzer=text_process)
X_train = vectorizer.fit_transform(train_data['Context'])
def questionQuestion(question,k=5):
    X_question = vectorizer.transform([question])
    similarities = cosine_similarity(X_question, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [8]:
# Test méthode 1 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [9]:
# Méthode 1 : Word2Vec
phrases_train = train_data["Context"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=1, workers=4)
def vectoriser_phrase(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
X_train_w2v = np.array([vectoriser_phrase(phrase) for phrase in train_data["Context"]])
def questionQuestion_w2v(question,k=5):
    vecteur_question = vectoriser_phrase(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list


In [10]:
# Test méthode 1 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [11]:
# Méthode 1 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
question_bert = model.encode(train_data["Context"].tolist())
def questionQuestion_bert(question,k=5):
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    list = []
    for idx in top_k_indices:
        list.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return list

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# Test méthode 1 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [13]:
# Méthde 2 : TF-IDF 
vectorizer = TfidfVectorizer(analyzer=text_process)
X_train = vectorizer.fit_transform(train_data['Response'])
def questionQuestion2(question,k=5):
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, X_train).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    list = []
    for idx in top_indices:
        list.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return list

In [14]:
# Test méthode 2 : TF-IDF
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [15]:
# Méthode 2 : Word2Vec
phrases_train = train_data["Response"].apply(text_process).tolist()
model_w2v = Word2Vec(sentences=phrases_train, vector_size=100, window=5, min_count=2)
def vectoriser_phrase2(phrase):
    mots = text_process(phrase)
    vecteurs = [model_w2v.wv[mot] for mot in mots if mot in model_w2v.wv]
    if vecteurs:
        return np.mean(vecteurs, axis=0)
    else:
        return np.zeros(model_w2v.vector_size)
X_train_w2v = np.array([vectoriser_phrase2(phrase) for phrase in train_data["Response"]])
def questionQuestion2_w2v(question,k=5):
    vecteur_question = vectoriser_phrase2(question).reshape(1, -1)
    similarities = cosine_similarity(vecteur_question, X_train_w2v).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = []
    for idx in top_indices:
        results.append((train_data.iloc[idx]["Response"], similarities[idx]))
    return results


In [16]:
# Test méthode 2 : Word2Vec
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_w2v(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [17]:
# Méthode 2 : Bert
model = SentenceTransformer('all-MiniLM-L6-v2')
question_bert = model.encode(train_data["Response"].tolist())
def questionQuestion2_bert(question,k=5):
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,question_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[::-1][:k]
    res = []
    for idx in top_k_indices:
        res.append((train_data.iloc[idx]["Response"], cosine_similarities[idx]))
    return res

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# Test méthode 2 : Bert
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top response:")
    list = questionQuestion2_bert(question)
    for predicted_response, score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {predicted_response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This is a fantastic question. In one sentence, I would say the following:Recognize that while you and your partner probably have common interests and areas of commonality, you are separate people, each with different wants, wishes, and desires – if you consider a diagram of two overlapping circles, they may share perhaps a third of the circle with overlap to indicate commonality (could be more or less) and then there are parts of the circles that are not overlapping, indicating separate interestsAs for ways that may strengthen any relationship, even the great ones, this is what came to mind. There are certainly more specific unique answers or elements for different people as far as the details, but here are some general ideas:Try to have at least 15 minutes a week where you are spending time together and not problem-solvingRealize that listening to your partner does not mean that you are agreeing with them, it just means th

In [25]:
X = model.encode(test_data['Response'].tolist())
def eval_mrr_bert(predictions):
    mrr_total = 0
    for i in range(len(test_data)):
        actualResponse = X[i].reshape(1, -1)
        predictedResponse = model.encode([predictions[i]])[0].reshape(1, -1)
        similarity = cosine_similarity(predictedResponse, actualResponse)[0]
        score = similarity.max()
        print(score)
        if score > 0:
            mrr_total += score
    return mrr_total / len(test_data)
predictions = []
for question in test_data['Context']:
    predicted_response = questionQuestion_bert(question)[0][0]
    predictions.append(predicted_response)
res = eval_mrr_bert(predictions)
print(f"MRR: {res:.4f}")

0.28320834
0.46932787
0.39413053
0.3371685
0.420528
0.30396378
0.29161763
0.5050529
0.27739853
0.56201285
MRR: 0.3844
